# PubMed Evidence Assistant - Notebook 3: serve it (API + public demo) on a free Colab T4

This runs the real service from the repo: **FastAPI** (`/ask`, `/feedback`, `/stats`, `/health`) in a background thread, the local Qwen model on the T4, and the **Gradio** demo with a public share link (free, no account).

**Before you start**
1. Notebooks 01 and 02 must have been run, so `MyDrive/pubmed_rag/results/` contains the `score_train_rag_whole_*.jsonl` files (they are the calibrator's training data).
2. Put the repo folder in Drive at `MyDrive/pubmed_rag/pubmed-evidence-assistant/` (upload the folder, or `git clone` it there).
3. Runtime -> T4 GPU.

The public link only lives while this notebook runs. That is fine for recording a demo video or showing it in an interview: open the notebook, run all cells, share the link.

In [ ]:
!pip -q install fastapi uvicorn rank_bm25 gradio requests datasets sentence-transformers accelerate scikit-learn

In [ ]:
import json, os, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/pubmed_rag")
REPO, RESULTS = BASE / "pubmed-evidence-assistant", BASE / "results"
assert REPO.exists(), f"Put the repo folder at {REPO}"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "app"))
os.chdir(REPO)
INDEX = REPO / "index"
print("repo:", REPO)

## 1. Build the index (once)

Downloads PubMedQA, embeds ~5,000 abstracts with bge-base on the GPU and writes `index/` into the repo folder on Drive. This is the **same corpus as the evaluation** (1,000 labelled abstracts + 4,000 distractors). Skipped if it already exists.

In [ ]:
from pubmed_rag.build import build_index

if not (INDEX / "embeddings.npy").exists():
    build_index(INDEX, n_distractors=4000)
meta = json.loads((INDEX / "meta.json").read_text())
print({k: meta[k] for k in ("n_docs", "n_questions", "embed_model", "built_at")})

## 2. Fit the calibrated decision head (once)

Uses the 300 training-question scores from Notebook 02 (`rag_whole` setup: whole abstracts, dense, k=5, i.e. exactly what the service does). Both variants are saved; the service uses `unweighted` (best accuracy on dev) unless you set `PMRAG_CALIBRATOR_VARIANT=balanced`.

In [ ]:
from pubmed_rag.calibrate import fit_from_jsonl

if not (INDEX / "calibrator.json").exists():
    files = sorted(RESULTS.glob("score_train_rag_whole_*.jsonl"), key=lambda p: p.stat().st_mtime)
    assert files, f"No score_train_rag_whole_*.jsonl in {RESULTS}. Run Notebook 02 (Part B) first."
    payload = fit_from_jsonl(files[-1:], INDEX / "calibrator.json")
    print("fitted on", payload["n_train"], "rows from", files[-1].name)
else:
    print("calibrator.json already exists")

## 3. Load the service (model download: a few minutes the first time)

In [ ]:
from pubmed_rag.api import build_assistant, create_app
from pubmed_rag.config import Settings

settings = Settings(index_dir=INDEX, backend="local", retrieval_mode="dense", k=5, log_dir=REPO / "logs")
assistant = build_assistant(settings)
print("backend:", assistant.backend.name, "| retrieval:", assistant.mode, "| calibrated:", assistant.calibrator is not None,
      "| corpus:", len(assistant.retriever.corpus))

### Sanity check: does the service reproduce the notebook's retrieval numbers?

Notebook 01 measured dense whole-abstract retrieval at hit@5 about 0.99 on all 1,000 questions. If the service's index or embedder were wrong, this number would collapse. It is a regression check for the deployment, not a new result.

In [ ]:
from pubmed_rag.evaluate import retrieval_metrics

questions = [json.loads(l) for l in (INDEX / "questions.jsonl").read_text().splitlines() if l.strip()]
m = retrieval_metrics(assistant.retriever, questions, "dense", k=5)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in m.items()})
assert m["hit_at_k"] > 0.97, "retrieval looks broken: check that index/ was built with the same embed model"

## 4. Start the API in the background and smoke-test it

In [ ]:
import requests
from pubmed_rag.serve import start_background

server, thread = start_background(create_app(settings, assistant), port=8000)
API = "http://127.0.0.1:8000"
print(requests.get(f"{API}/health").json())

r = requests.post(f"{API}/ask", json={"question": "Does high blood pressure reduce the risk of chronic low back pain?"}).json()
print(r["decision"], {k: round(v, 3) for k, v in r["probabilities"].items()}, "| top abstract:", r["evidence"][0]["doc_id"])
print("timings (ms):", r["timings_ms"])

## 5. Launch the public demo

The cell below stays running (that is what keeps the link alive). Copy the `https://....gradio.live` URL it prints. To stop: interrupt the cell.

In [ ]:
from demo import build_demo

demo = build_demo(API)
demo.launch(share=True, debug=True)

## 6. After a demo session: what the service recorded (AIOps)

Run these after interrupting the cell above. The API has been running in the background, so `/stats` and the logs reflect everything you did in the demo. Questions are logged only as hashes unless you set `log_questions=True`.

In [ ]:
print(requests.get(f"{API}/stats").json())
print(open(REPO / "logs" / "requests.jsonl").read().splitlines()[-1][:300])
fb = REPO / "logs" / "feedback.jsonl"
print("feedback rows:", len(fb.read_text().splitlines()) if fb.exists() else 0)

## Record the demo

Screen-record about 60-90 seconds: ask one of the example questions, show the verdict, the probabilities and the cited PubMed abstracts, click a PubMed link, use the feedback button, then show `/stats`. Add the video (or a GIF) to the repo README.